# 游戏 NPC 角色扮演云服务（Kaggle）

在 Kaggle 免费 GPU 上部署 **Qwen2.5-7B-Instruct** 角色扮演 API。

**运行步骤**：
1. 点右上角 **Settings** → **Accelerator** 选 **GPU T4 x2**（免费）
2. 依次运行下面所有 cell（Ctrl+Enter 逐个）
3. 最后一个 cell 打印公网 URL（trycloudflare.com），复制它
4. 本机运行: `python scripts/cloud_engine.py --url <URL> --test`
5. WebUI 接入: `python webui.py --engine cloud --cloud_url <URL>`

In [ ]:
# 1. 检查 GPU + 装依赖
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory/1e9, 'GB')
else:
    print('!!! 没有 GPU！请到 Settings -> Accelerator 选择 GPU T4 x2，然后重启会话')

!pip install -q transformers accelerate gradio cloudflared modelscope

In [ ]:
# 2. 加载 Qwen2.5-7B（ModelScope，国内快）
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = 'Qwen/Qwen2.5-7B-Instruct'

print('下载模型（首次约 3 分钟）...')
try:
    # 优先 ModelScope
    from modelscope import AutoModelForCausalLM as MSModel, AutoTokenizer as MSTok
    tok = MSTok.from_pretrained(model_name, trust_remote_code=True)
    model = MSModel.from_pretrained(model_name, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
except Exception as e:
    print('ModelScope 失败，用 HuggingFace:', e)
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)

model.eval()
print('模型加载完成! 显存:', torch.cuda.memory_allocated()/1e9, 'GB')

In [ ]:
# 3. 角色扮演生成函数 + 测试
def generate(character, scene, state, player_input):
    sys_p = (
        f"你是游戏NPC「{character['name']}」，{character['identity']}。"
        f"性格：{character['personality']}。说话风格：{character['speech_style']}。\n"
        f"请始终保持角色，回复要短（≤40字），绝不承认自己是AI。"
        f"只输出对玩家说的台词，禁止旁白和内心戏。"
    )
    user_p = (
        f"[当前状态]\n场景：{scene}\n好感度：{state.get('好感度',0)}\n"
        f"任务：{state.get('任务','无')}\n\n[玩家]\n{player_input}"
    )
    msgs = [
        {'role': 'system', 'content': sys_p},
        {'role': 'user', 'content': user_p},
    ]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=64, temperature=0.7, top_p=0.9,
            do_sample=True, pad_token_id=tok.pad_token_id)
    reply = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return reply.strip()

# 测试
test_char = {'name':'艾拉','identity':'流浪商人','personality':'警惕、爱钱、嘴硬心软','speech_style':'短句、带刺'}
print('测试:', generate(test_char, '集市摊位', {'好感度':0}, '你是谁？'))

In [ ]:
# 4. Gradio API 服务（本机 WebUI 会调用这个）
import gradio as gr

CHARACTERS = {
    'aila': {'name':'艾拉','identity':'流浪商人','personality':'警惕、爱钱、嘴硬心软','speech_style':'短句、带刺'},
    'bruno': {'name':'布鲁诺','identity':'酒馆老板','personality':'豪爽、爱吹牛、消息灵通','speech_style':'简短、粗犷、喜欢夸张'},
    'kara': {'name':'卡拉','identity':'铁匠','personality':'沉默寡言、手艺好','speech_style':'话少、直接'},
    'orin': {'name':'奥林','identity':'年轻守卫','personality':'认真负责、天真','speech_style':'规矩、热情'},
    'morgan': {'name':'摩根','identity':'老猎人','personality':'话少、经验丰富','speech_style':'简短、直接'},
    'luna': {'name':'露娜','identity':'女巫学徒','personality':'好奇心重、跳脱','speech_style':'活泼、跳跃'},
    'victor': {'name':'维克托','identity':'反派手下','personality':'傲慢、嘴毒','speech_style':'刻薄、阴阳怪气'},
    'elda': {'name':'艾尔达','identity':'村长夫人','personality':'慈祥、爱八卦','speech_style':'温和、唠叨'},
}

def chat_api(cid, player_input, scene, affection):
    char = CHARACTERS.get(cid, CHARACTERS['aila'])
    return generate(char, scene, {'好感度': affection}, player_input)

demo = gr.Interface(
    fn=chat_api,
    inputs=[
        gr.Dropdown(list(CHARACTERS.keys()), value='aila', label='角色'),
        gr.Textbox(label='玩家输入'),
        gr.Dropdown(['夜晚营地','热闹的酒馆','集市摊位','村口老树下','铁匠铺门口'], value='集市摊位', label='场景'),
        gr.Slider(-100, 100, 0, label='好感度'),
    ],
    outputs=gr.Textbox(label='NPC回复'),
    title='NPC Roleplay API (Qwen2.5-7B)',
)
demo.launch(server_name='0.0.0.0', server_port=7860, quiet=True, prevent_thread_lock=True)
print('Gradio 已启动在 7860')

In [ ]:
# 5. cloudflared 隧道暴露公网 URL（保持本 cell 运行！）
import subprocess, time, re

print('启动 cloudflared 隧道...')
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url = None
for line in proc.stdout:
    if 'trycloudflare.com' in line:
        m = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
        if m:
            url = m.group(1)
            break

print('\n' + '='*60)
print(' 你的服务公网 URL:')
print(' ' + url)
print('='*60)
print('在本机运行: python scripts/cloud_engine.py --url ' + url + ' --test')
print('然后: python webui.py --engine cloud --cloud_url ' + url)
print('（保持本 cell 运行，勿中断）')

while True:
    time.sleep(60)